# Hotel Booking Cancellation: ML Preprocessing

This notebook mirrors the supplied preprocessing workflow while adapting it to hotel-booking cancellation. It profiles the data, defines a leakage-safe feature set, splits before fitting transformations, imputes missing values, scales numerical variables, one-hot encodes nominal variables, and balances only the training set.


## Data understanding


| Column | Meaning |
| --- | --- |
| `hotel` | Hotel type: City Hotel or Resort Hotel. |
| `is_canceled` | Target flag: 1 if the booking was canceled, otherwise 0. |
| `lead_time` | Days between booking creation and arrival. |
| `arrival_date_year` | Scheduled arrival year. |
| `arrival_date_month` | Scheduled arrival month name. |
| `arrival_date_week_number` | Week number of the scheduled arrival date. |
| `arrival_date_day_of_month` | Day of month of the scheduled arrival date. |
| `stays_in_weekend_nights` | Number of weekend nights booked. |
| `stays_in_week_nights` | Number of weekday nights booked. |
| `adults` | Number of adults in the booking. |
| `children` | Number of children in the booking. |
| `babies` | Number of babies in the booking. |
| `meal` | Booked meal plan. |
| `country` | Guest country code. |
| `market_segment` | Market segment through which the booking was made. |
| `distribution_channel` | Booking distribution channel. |
| `is_repeated_guest` | Flag showing whether the guest booked before. |
| `previous_cancellations` | Number of earlier bookings canceled by the guest. |
| `previous_bookings_not_canceled` | Number of earlier bookings not canceled by the guest. |
| `reserved_room_type` | Reserved room-type code. |
| `assigned_room_type` | Room-type code ultimately assigned. |
| `booking_changes` | Number of changes made to the booking. |
| `deposit_type` | Deposit category. |
| `agent` | Booking-agent identifier. |
| `company` | Company or entity identifier associated with the booking. |
| `days_in_waiting_list` | Days the booking remained on a waiting list. |
| `customer_type` | Customer category such as Transient or Contract. |
| `adr` | Average daily rate, calculated as lodging revenue divided by occupied room nights. |
| `required_car_parking_spaces` | Number of requested parking spaces. |
| `total_of_special_requests` | Number of special requests made by the guest. |
| `reservation_status` | Final reservation status. This is an outcome field. |
| `reservation_status_date` | Date on which the final reservation status was set. |


## Modeling objective

Predict `is_canceled` at booking time. Fields only known after later booking activity or after the final outcome are excluded before modeling. This prevents the preprocessing output from leaking the answer into the features.


## Data loading


In [ ]:
import calendar
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.utils import resample

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
sns.set(style="whitegrid", context="notebook")


In [ ]:
from pathlib import Path

def find_data_file(filename="hotel_bookings.csv"):
    candidates = [
        Path(filename),
        Path.cwd() / filename,
        Path.cwd().parent / filename,
        Path.home() / "Downloads" / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    searched = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError("Could not find {}. Searched:\n{}".format(filename, searched))

DATA_PATH = find_data_file()
raw = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH.name)
print("Shape:", raw.shape)
raw.head()


## Data exploration


In [ ]:
raw.info()


In [ ]:
raw.describe().T.round(2)


In [ ]:
raw.describe(include=["object"]).T


### Duplicates and missing values


In [ ]:
quality_summary = pd.Series(
    {
        "rows": len(raw),
        "columns": raw.shape[1],
        "exact_duplicate_rows": int(raw.duplicated().sum()),
        "duplicate_share": raw.duplicated().mean(),
        "columns_with_missing_values": int(raw.isna().any().sum()),
    }
)
quality_summary


In [ ]:
missing = pd.DataFrame(
    {
        "missing_count": raw.isna().sum(),
        "missing_percent": raw.isna().mean().mul(100).round(2),
    }
)
missing.loc[missing["missing_count"] > 0].sort_values("missing_percent", ascending=False)


### Target distribution


In [ ]:
target_distribution = (
    raw["is_canceled"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Not canceled", 1: "Canceled"})
    .rename("bookings")
    .to_frame()
)
target_distribution["share"] = target_distribution["bookings"] / target_distribution["bookings"].sum()
display(target_distribution)

ax = sns.barplot(
    x=target_distribution.index,
    y=target_distribution["bookings"],
    palette=["#59A14F", "#E15759"],
)
ax.set_title("Target distribution")
ax.set_xlabel("")
ax.set_ylabel("Bookings")
plt.tight_layout()
plt.show()


## Feature engineering


In [ ]:
model_df = raw.copy()

# Remove exact duplicates before splitting so identical rows cannot appear in both sets.
# Because the dataset has no booking ID, this is a modeling choice rather than a claim
# that repeated rows are definitely duplicate business events.
model_df = model_df.drop_duplicates().copy()

# Remove records that do not represent a valid guest booking and the one negative ADR row.
guest_count = model_df["adults"] + model_df["children"].fillna(0) + model_df["babies"]
model_df = model_df.loc[(guest_count > 0) & (model_df["adr"] >= 0)].copy()

model_df["total_nights"] = (
    model_df["stays_in_weekend_nights"] + model_df["stays_in_week_nights"]
)
model_df["total_guests"] = (
    model_df["adults"] + model_df["children"].fillna(0) + model_df["babies"]
)
model_df["has_children"] = (
    (model_df["children"].fillna(0) + model_df["babies"]) > 0
).astype(int)

month_number = {
    month: index for index, month in enumerate(list(calendar.month_name)[1:], start=1)
}
model_df["arrival_month_number"] = model_df["arrival_date_month"].map(month_number)
model_df["arrival_month_sin"] = np.sin(2 * np.pi * model_df["arrival_month_number"] / 12)
model_df["arrival_month_cos"] = np.cos(2 * np.pi * model_df["arrival_month_number"] / 12)

print("Rows after modeling cleanup:", len(model_df))
model_df[["total_nights", "total_guests", "has_children", "arrival_month_sin", "arrival_month_cos"]].head()


## Remove leakage and identifier-like columns


In [ ]:
target = "is_canceled"

leakage_columns = [
    "reservation_status",       # Directly reveals the final outcome.
    "reservation_status_date",  # Recorded when final status is set.
    "assigned_room_type",       # May only be known near check-in.
    "booking_changes",          # Can accumulate after the initial booking.
    "days_in_waiting_list",     # Can evolve after the initial booking.
]
identifier_columns = [
    "agent",                    # High-cardinality external identifier.
    "company",                  # High-cardinality and mostly missing identifier.
]
redundant_columns = [
    "arrival_date_month",        # Replaced by cyclical month features.
    "arrival_month_number",      # Used only to calculate sine/cosine features.
    "stays_in_weekend_nights",   # Replaced by total_nights.
    "stays_in_week_nights",      # Replaced by total_nights.
    "adults", "children", "babies",  # Replaced by total_guests/has_children.
]

columns_to_drop = leakage_columns + identifier_columns + redundant_columns
X = model_df.drop(columns=[target] + columns_to_drop)
y = model_df[target].astype(int)

print("Feature matrix before encoding:", X.shape)
print("Dropped columns:", columns_to_drop)
X.head()


## Train/test split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

split_summary = pd.DataFrame(
    {
        "train_count": y_train.value_counts().sort_index(),
        "test_count": y_test.value_counts().sort_index(),
        "train_share": y_train.value_counts(normalize=True).sort_index(),
        "test_share": y_test.value_counts(normalize=True).sort_index(),
    }
)
split_summary.index = ["Not canceled", "Canceled"]
split_summary


All learned preprocessing is fitted on `X_train` only. The test set is transformed using those fitted rules, which keeps evaluation data out of imputation, scaling, and category discovery.


## Define numerical, binary, and categorical columns


In [ ]:
binary_features = ["is_repeated_guest", "has_children"]
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()
numeric_features = [
    column
    for column in X_train.select_dtypes(include=["number"]).columns
    if column not in binary_features
]

print("Numeric features ({}):".format(len(numeric_features)), numeric_features)
print("Binary features ({}):".format(len(binary_features)), binary_features)
print("Categorical features ({}):".format(len(categorical_features)), categorical_features)


### Missing values in the training split


In [ ]:
train_missing = pd.DataFrame(
    {
        "missing_count": X_train.isna().sum(),
        "missing_percent": X_train.isna().mean().mul(100).round(2),
    }
)
train_missing.loc[train_missing["missing_count"] > 0].sort_values(
    "missing_percent", ascending=False
)


## Imputation, robust scaling, and nominal encoding


In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ]
)

binary_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="most_frequent"))]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                drop=None,
                sparse=True,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("binary", binary_pipeline, binary_features),
        ("categorical", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)

print("Training matrix:", X_train_ready.shape)
print("Test matrix:", X_test_ready.shape)
print("Sparse output:", sparse.issparse(X_train_ready))


## Encoded feature names


In [ ]:
onehot = preprocessor.named_transformers_["categorical"].named_steps["onehot"]
if hasattr(onehot, "get_feature_names_out"):
    encoded_names = onehot.get_feature_names_out(categorical_features).tolist()
else:
    encoded_names = onehot.get_feature_names(categorical_features).tolist()

feature_names = numeric_features + binary_features + encoded_names
print("Total prepared features:", len(feature_names))
feature_names[:30]


## Verify the prepared matrices


In [ ]:
def count_missing_in_matrix(matrix):
    values = matrix.data if sparse.issparse(matrix) else np.asarray(matrix)
    return int(np.isnan(values).sum())

verification = pd.Series(
    {
        "train_rows": X_train_ready.shape[0],
        "test_rows": X_test_ready.shape[0],
        "prepared_features": X_train_ready.shape[1],
        "feature_name_count": len(feature_names),
        "train_missing_values": count_missing_in_matrix(X_train_ready),
        "test_missing_values": count_missing_in_matrix(X_test_ready),
        "train_target_missing": int(y_train.isna().sum()),
        "test_target_missing": int(y_test.isna().sum()),
    }
)
verification


## Handle class imbalance

The reference notebook uses oversampling. Here, the minority class is randomly oversampled after transformation and only within the training split. This preserves valid one-hot values and avoids exposing test rows to the balancing step.


In [ ]:
y_train_array = y_train.to_numpy()
class_counts = pd.Series(y_train_array).value_counts()
majority_class = class_counts.idxmax()
minority_class = class_counts.idxmin()

majority_indices = np.flatnonzero(y_train_array == majority_class)
minority_indices = np.flatnonzero(y_train_array == minority_class)

minority_upsampled = resample(
    minority_indices,
    replace=True,
    n_samples=len(majority_indices),
    random_state=42,
)
balanced_indices = np.concatenate([majority_indices, minority_upsampled])
balanced_indices = np.random.RandomState(42).permutation(balanced_indices)

X_train_balanced = X_train_ready[balanced_indices]
y_train_balanced = y_train_array[balanced_indices]

pd.DataFrame(
    {
        "before": pd.Series(y_train_array).value_counts().sort_index(),
        "after": pd.Series(y_train_balanced).value_counts().sort_index(),
    },
    index=[0, 1],
).rename(index={0: "Not canceled", 1: "Canceled"})


## Final preprocessing outputs


In [ ]:
final_outputs = pd.Series(
    {
        "X_train_balanced_shape": X_train_balanced.shape,
        "X_test_ready_shape": X_test_ready.shape,
        "y_train_balanced_length": len(y_train_balanced),
        "y_test_length": len(y_test),
        "prepared_feature_count": len(feature_names),
        "training_missing_values": count_missing_in_matrix(X_train_balanced),
        "test_missing_values": count_missing_in_matrix(X_test_ready),
    }
)
final_outputs


The objects ready for downstream modeling are:

- `preprocessor`: fitted transformation pipeline.
- `X_train_balanced`, `y_train_balanced`: balanced, prepared training data.
- `X_test_ready`, `y_test`: untouched test target with features transformed by training-only rules.
- `feature_names`: prepared feature labels.

Keep the test set unbalanced to measure performance on the original class distribution.
